# Target Detection via Denoising Score Matching

Reproduces every experiment in the paper on Pavia-University:
1. **IID single-class** (Fig. 2) — AMF, GMM-Levin, L-DART, **DART**, L-LRao, LRao
2. **IID multi-class** (Fig. 3)
3. **Spatial / correlated** (Table 1) — adds AMF-local, DART-CFAR, **DARTS**, DARTS-CFAR

Runs on Colab or locally. Edit the **Algorithm hyperparameters** cell to tune
detector behaviour (the values shown are the paper settings). `QUICK = True` for
a fast smoke test.

## Setup (Colab or local)

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
GIT_URL    = ''                  # e.g. 'https://github.com/<user>/<repo>.git'  ('' = files already present)
GIT_BRANCH = 'reviewer-release'  # branch holding this clean release

def have_repo():
    return os.path.isfile('src/iid.py') and os.path.isfile('data/pavia-u.mat')

if not have_repo():
    if GIT_URL:
        subprocess.run(['git','clone','--depth','1','-b',GIT_BRANCH,GIT_URL,'repo'], check=True)
        os.chdir('repo')
    for cand in ('SDSM','repo','final-paper-experiment'):
        if not have_repo() and os.path.isdir(cand):
            os.chdir(cand)
assert have_repo(), ('Project files not found. Set GIT_URL above (clone uses branch '
                     f'{GIT_BRANCH!r}), or upload the project folder/.zip in Colab, then re-run.')
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)
sys.path.insert(0, os.getcwd())
print('OK | cwd =', os.getcwd())

In [ ]:
import yaml, glob
import torch
from IPython.display import Image, display

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
QUICK  = True          # True = fast smoke test; False = full paper settings
print('device =', DEVICE, '| QUICK =', QUICK)

# src.iid sets the matplotlib 'Agg' (headless) backend on import, so we display
# the saved PNGs with IPython.display (backend-independent), not plt.show().
_IID_FIGS = ['roc_at_nmax.png','auc_vs_n.png','pauc_vs_n.png','pd_at_fa_vs_n.png',
             'auc_vs_rho.png','pdet_at_pfa_vs_rho.png','roc_auc_bar_nmax.png']

def show_figs(run_dir, names=None):
    fdir = os.path.join(run_dir,'figures'); names = names or _IID_FIGS; shown=0
    for nm in names:
        p=os.path.join(fdir,nm)
        if os.path.exists(p): display(Image(filename=p)); shown+=1
    for lp in sorted(glob.glob(os.path.join(fdir,'loss_curves_*.png'))):
        display(Image(filename=lp)); shown+=1
    if shown==0: print('(no figures in',fdir,')')

# QUICK shrinks ONLY the training budget (epochs / seeds); it keeps the full
# paper sweep (n_train_list / rho_list) so the plots have the right shape.
def quicken(cfg, spatial=False):
    if not QUICK: return cfg
    if spatial: cfg.update(dsm_epochs=20, nmlp_epochs=10)
    else: cfg.update(seed=42, dsm_epochs=80, lrao_epochs=40, test_size=800)  # full sweep kept
    return cfg

## Algorithm hyperparameters — edit these

Detection-algorithm knobs only (they change *what each detector computes*). The
values are the paper settings; some differ between single and multi (e.g.
`lfi_sigma_cutoff`, `lrao_whiten_eig_floor`), so they live in the per-experiment
dicts. Training settings (optimizer, `*_epochs`, early stopping, architecture)
stay in `configs/*.yaml`.

In [ ]:
# ---- IID: shared knobs (same for single + multi) ----
ALG_IID = dict(
    dsm_sigma_rho    = 0.1,      # DART / L-DART score-matching noise level rho
    whiten_eig_floor = 0.0,      # ZCA floor for DART/L-DART (0 -> auto 1e-5*lambda_max)
    lfi_delta_theta  = 0.01,     # LRao / L-LRao finite-difference step (Jacobian)
    lfi_detach_sigma = True,     # LRao / L-LRao: detach Sigma in the objective (stable)
)
# per-experiment knobs (these DIFFER between single and multi in the paper)
ALG_IID_SINGLE = dict(ALG_IID, gmm_K=2, lfi_sigma_cutoff=1e-6, lrao_whiten_eig_floor=1e-10)
ALG_IID_MULTI  = dict(ALG_IID, gmm_K=9, lfi_sigma_cutoff=1e-3, lrao_whiten_eig_floor=1e-5)

# ---- Spatial / correlated background (paper Table 1 values) ----
ALG_SPATIAL = dict(
    n_budget             = 2000,      # contiguous side-crop of the train box (None = full box)
    k                    = 5,         # k x k spatial window
    nmlp_K               = 7,         # latent-nearest neighbours pooled
    dsm_sigma_rho        = 0.1,
    whiten_eig_floor     = 1e-5,
    pfa_target           = 0.05,      # CFAR threshold target false-alarm rate
    cfar_lam             = 0.1,       # local -> global Fisher shrinkage (0 local, 1 global)
    cfar_fisher_use_topk = False,
    sdsm_cfar_window     = None,
    sdsm_cfar_guard      = 1,
    dsm_cfar_window      = 11,
    dsm_cfar_guard       = 3,
    amf_local_window     = 15,        # AMF-local SCM window (>= D samples)
    local_scm_loading    = 1e-18,
    baseline_eig_floor   = 1e-18,
    gmm_K                = 9,
    gmm_steps            = 50,
    scenario_index       = 4,         # 0-3 manual boxes; 4+ random boxes
    foreign_class        = 7,         # planted foreign-target class (bitumen)
    amplitude            = 0.15,
    target_fraction      = 0.10,
)
print('algorithm knobs ready')

## 1. IID — single-class background (Fig. 2)

In [ ]:
import src.iid as iid
cfg = yaml.safe_load(open('configs/iid_single.yaml')); cfg['device']=DEVICE
cfg.update(ALG_IID_SINGLE); cfg = quicken(cfg)
run_dir_single, _ = iid.run_iid(cfg, mode='single')
show_figs(run_dir_single)

## 2. IID — multi-class background (Fig. 3)

In [ ]:
cfg = yaml.safe_load(open('configs/iid_multi.yaml')); cfg['device']=DEVICE
cfg.update(ALG_IID_MULTI); cfg = quicken(cfg)
run_dir_multi, _ = iid.run_iid(cfg, mode='multi')
show_figs(run_dir_multi)

## 3. Spatial — correlated background (Table 1)

Trains the global DART and the spatially adapted DARTS on disjoint train/test
boxes (with `n_budget` secondary pixels), compares all detectors, prints the
summary table. `QUICK=False` averages over seeds via `run_multiseed`.

In [ ]:
import src.spatial as spatial
cfg = yaml.safe_load(open('configs/spatial.yaml')); cfg['device']=DEVICE
cfg.update(ALG_SPATIAL); cfg = quicken(cfg, spatial=True)
if QUICK:
    parent = spatial.run_from_cfg(cfg, dry_run=False)
    spatial.show_plots_from_dir(parent, sub='foreign', inline=True)
else:
    parent = spatial.run_multiseed(cfg, seeds=(42,43,44,45,46))
    spatial.show_multiseed(parent, inline=True)
print('spatial results ->', parent)

## CFAR ablation (optional)

Sweep `cfar_lam` (local->global Fisher shrinkage) for the DART-CFAR / DARTS-CFAR
rows of Table 1. Pure scoring knob, no retraining.

In [ ]:
for lam in (0.0, 0.15, 0.3, 0.5, 0.7, 1.0):
    c = dict(cfg, cfar_lam=lam,
             active_detectors=['DART-CFAR','DARTS-CFAR','DARTS','AMF','GMM-Levin'])
    rd = spatial.run_from_cfg(c, dry_run=False)
    print(f'cfar_lam={lam} ->', rd)